# Labor Market Indicator Research

## Purpose

This notebook evaluates county-level labor-market indicators that may better
represent economic insecurity than the official unemployment rate for possible
use in the Economic Stress Index (ESI).

The original ESI combined poverty, unemployment, and severe housing burden.
Sensitivity analysis of alternative normalization methods raised a conceptual
concern: while high unemployment clearly indicates economic stress, low
unemployment does not necessarily indicate economic stability.

The official unemployment rate does not fully capture conditions such as
low-wage employment, insufficient hours, multiple-job holding, lack of benefits,
or people who have left the labor force. A county may therefore have low
unemployment while households continue to experience substantial economic
hardship.

## Research Question

**Can a county-level labor-market indicator better capture economic insecurity
among people who are technically employed, rather than treating low unemployment
as evidence of economic stability?**

## Candidate Measures

Potential indicators to investigate include:

- Labor force participation rate
- Employment-to-population ratio
- Median earnings
- Full-time/full-year employment
- Working poverty or low-earnings measures
- Other nationally consistent county-level measures of employment quality or
  economic security

## Evaluation Criteria

Candidate indicators will be evaluated for:

1. Conceptual relevance to economic stress
2. Consistent availability across U.S. counties and county equivalents
3. Data quality and interpretability
4. Relationship with poverty and severe housing burden
5. Relationship with food insecurity
6. Relationship with median household income
7. Information added beyond the official unemployment rate
8. Suitability for a transparent, public-facing Economic Stress Index

## Stage 1 — Candidate Labor-Market Measures

The official unemployment rate measures joblessness among people participating
in the labor force, but it does not directly measure employment quality,
earnings adequacy, work intensity, or economic security among employed workers.

The following county-level measures will therefore be investigated as potential
alternatives or complements to unemployment.

### Candidate 1 — Labor Force Participation Rate
Measures the share of the working-age population participating in the labor
force.

**Potential strength:** Captures labor-market attachment beyond unemployment.

**Potential limitation:** Low participation may reflect retirement, education,
caregiving, disability, or other circumstances that are not necessarily
economic distress.

### Candidate 2 — Employment-to-Population Ratio
Measures the share of the working-age population that is employed.

**Potential strength:** Does not depend on whether a non-employed person is
actively searching for work.

**Potential limitation:** Employment alone does not indicate adequate wages,
hours, benefits, or job stability.

### Candidate 3 — Median Earnings
Measures earnings among workers rather than whether individuals are simply
classified as employed.

**Potential strength:** More directly reflects the economic value of employment.

**Potential limitation:** May overlap conceptually and statistically with
poverty and household income and does not directly measure work stability.

### Candidate 4 — Full-Time / Year-Round Work
Measures the extent to which employment provides sustained work across the year.

**Potential strength:** May distinguish stable employment from intermittent or
limited work.

**Potential limitation:** Part-time work may be voluntary, and the measure does
not directly capture wage adequacy.

### Research Principle

No candidate will be selected solely because it produces the strongest
correlation with food insecurity. The selected measure should represent a
distinct and defensible dimension of economic stress while adding meaningful
information beyond poverty and severe housing burden.

# Emerging hypothesis: 

## A single unemployment measure may be insufficient to represent labor-market economic stress. This analysis will therefore also evaluate whether multiple complementary labor indicators can support a separate Labor Stress measure capturing labor-market attachment, work stability, and earnings adequacy.

## Stage 2 — Prime-Age Full-Time, Year-Round Employment

The ACS 2024 5-Year Detailed Table B23027 reports full-time,
year-round work status by age.

For this analysis, prime working age is defined as ages 25–54,
combining the ACS age groups 25–44 and 45–54.

The initial measure calculates the percentage of the prime-age
population that worked full-time, year-round during the past
12 months.

This measure is being evaluated as an alternative indicator of
labor-market economic security because it captures sustained
employment rather than unemployment status alone.

In [1]:
from pathlib import Path
import os

import pandas as pd
import requests
from dotenv import load_dotenv

PROJECT_DIR = Path.cwd().parent
ENV_PATH = PROJECT_DIR / ".env"

load_dotenv(ENV_PATH)

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

print("Project directory:", PROJECT_DIR)
print("API key loaded:", CENSUS_API_KEY is not None)

Project directory: /Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer
API key loaded: True


In [15]:
print(PROJECT_DIR)

for path in (PROJECT_DIR / "data").rglob("*"):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

/Users/suzanneuvino/Desktop/Economic_Stress_Research/County_Explorer
data/.DS_Store
data/raw/Economic_Stress_US_Counties_Master.xlsx
data/raw/MMG2026_2024_Data_To_Share.xlsx


In [17]:
# Load canonical 32-column County Explorer master

MASTER_PATH = (
    PROJECT_DIR
    / "outputs"
    / "County_Explorer_Master_2026.xlsx"
)

master_final = pd.read_excel(
    MASTER_PATH,
    dtype={
        "FIPS": str,
        "State_FIPS": str,
        "County_FIPS": str
    }
)

master_final["FIPS"] = master_final["FIPS"].str.zfill(5)

print("Rows:", len(master_final))
print("Columns:", len(master_final.columns))
print("Unique FIPS:", master_final["FIPS"].nunique())
print("Duplicate FIPS:", master_final["FIPS"].duplicated().sum())

Rows: 3144
Columns: 32
Unique FIPS: 3144
Duplicate FIPS: 0


In [2]:
# ACS 2024 5-Year Detailed Table B23027
# Prime-age full-time/year-round employment

url = "https://api.census.gov/data/2024/acs/acs5"

params = {
    "get": ",".join([
        "NAME",
        "B23027_012E",  # Age 25-44 total
        "B23027_014E",  # Age 25-44 FT/YR
        "B23027_015E",  # Age 25-44 less than FT/YR
        "B23027_016E",  # Age 25-44 did not work
        "B23027_017E",  # Age 45-54 total
        "B23027_019E",  # Age 45-54 FT/YR
        "B23027_020E",  # Age 45-54 less than FT/YR
        "B23027_021E"   # Age 45-54 did not work
    ]),
    "for": "county:*",
    "in": "state:*",
    "key": CENSUS_API_KEY
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [3]:
# Convert Census API response to DataFrame

data = response.json()

labor_raw = pd.DataFrame(
    data[1:],
    columns=data[0]
)

print("Raw shape:", labor_raw.shape)
display(labor_raw.head())

Raw shape: (3222, 11)


,NAME,B23027_012E,B23027_014E,B23027_015E,B23027_016E,B23027_017E,B23027_019E,B23027_020E,B23027_021E,state,county
0,"Autauga County, Alabama",15816,10800,1945,3071,7926,4909,1108,1909,01,001
1,"Baldwin County, Alabama",57170,37067,11094,9009,31499,21129,5360,5010,01,003
2,"Barbour County, Alabama",6488,3047,1196,2245,2911,1351,575,985,01,005
3,"Bibb County, Alabama",6084,2993,1227,1864,3026,1669,527,830,01,007
4,"Blount County, Alabama",14167,9017,2593,2557,7646,4430,1310,1906,01,009


In [4]:
# Create 5-digit county FIPS

labor_raw["FIPS"] = (
    labor_raw["state"].str.zfill(2)
    + labor_raw["county"].str.zfill(3)
)

count_columns = [
    "B23027_012E",
    "B23027_014E",
    "B23027_015E",
    "B23027_016E",
    "B23027_017E",
    "B23027_019E",
    "B23027_020E",
    "B23027_021E"
]

labor_raw[count_columns] = (
    labor_raw[count_columns]
    .apply(pd.to_numeric, errors="coerce")
)

print("Rows:", len(labor_raw))
print("Unique FIPS:", labor_raw["FIPS"].nunique())
print("Duplicate FIPS:", labor_raw["FIPS"].duplicated().sum())
print("All FIPS 5 digits:", labor_raw["FIPS"].str.len().eq(5).all())

Rows: 3222
Unique FIPS: 3222
Duplicate FIPS: 0
All FIPS 5 digits: True


In [5]:
# Inspect state/territory FIPS counts

display(
    labor_raw.groupby("state")
    .size()
    .reset_index(name="County_Count")
    .tail(10)
)

,state,County_Count
42,47,95
43,48,254
44,49,29
45,50,14
46,51,133
47,53,39
48,54,55
49,55,72
50,56,23
51,72,78


In [6]:
# Exclude Puerto Rico to match the canonical county universe

labor_raw = labor_raw[
    labor_raw["state"] != "72"
].copy()

print("Rows:", len(labor_raw))
print("Unique FIPS:", labor_raw["FIPS"].nunique())
print("Duplicate FIPS:", labor_raw["FIPS"].duplicated().sum())

Rows: 3144
Unique FIPS: 3144
Duplicate FIPS: 0


In [7]:
# Combine ACS age groups into prime working age 25-54

labor_raw["Prime_Age_Pop"] = (
    labor_raw["B23027_012E"]
    + labor_raw["B23027_017E"]
)

labor_raw["Prime_Age_FTYR"] = (
    labor_raw["B23027_014E"]
    + labor_raw["B23027_019E"]
)

labor_raw["Prime_Age_Less_Than_FTYR"] = (
    labor_raw["B23027_015E"]
    + labor_raw["B23027_020E"]
)

labor_raw["Prime_Age_Did_Not_Work"] = (
    labor_raw["B23027_016E"]
    + labor_raw["B23027_021E"]
)

In [8]:
# Validate B23027 component counts

labor_raw["Status_Sum"] = (
    labor_raw["Prime_Age_FTYR"]
    + labor_raw["Prime_Age_Less_Than_FTYR"]
    + labor_raw["Prime_Age_Did_Not_Work"]
)

labor_raw["Status_Difference"] = (
    labor_raw["Prime_Age_Pop"]
    - labor_raw["Status_Sum"]
)

print("Status difference values:")
print(
    labor_raw["Status_Difference"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nCounties with non-zero difference:",
    (labor_raw["Status_Difference"] != 0).sum()
)

Status difference values:
Status_Difference
0    3144
Name: count, dtype: int64

Counties with non-zero difference: 0


In [9]:
# Calculate prime-age work-status percentages

labor_raw["Prime_Age_FTYR_Pct"] = (
    labor_raw["Prime_Age_FTYR"]
    / labor_raw["Prime_Age_Pop"]
    * 100
)

labor_raw["Prime_Age_Less_Than_FTYR_Pct"] = (
    labor_raw["Prime_Age_Less_Than_FTYR"]
    / labor_raw["Prime_Age_Pop"]
    * 100
)

labor_raw["Prime_Age_Did_Not_Work_Pct"] = (
    labor_raw["Prime_Age_Did_Not_Work"]
    / labor_raw["Prime_Age_Pop"]
    * 100
)

In [10]:
# Validate that work-status percentages sum to 100%

labor_raw["Work_Status_Pct_Sum"] = (
    labor_raw["Prime_Age_FTYR_Pct"]
    + labor_raw["Prime_Age_Less_Than_FTYR_Pct"]
    + labor_raw["Prime_Age_Did_Not_Work_Pct"]
)

print(
    labor_raw["Work_Status_Pct_Sum"]
    .describe()
    .round(6)
)

print(
    "\nCount not approximately 100%:",
    (~labor_raw["Work_Status_Pct_Sum"].between(99.999, 100.001)).sum()
)

count    3143.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
Name: Work_Status_Pct_Sum, dtype: float64

Count not approximately 100%: 1


In [11]:
# Inspect distributions of prime-age work status

display(
    labor_raw[
        [
            "Prime_Age_FTYR_Pct",
            "Prime_Age_Less_Than_FTYR_Pct",
            "Prime_Age_Did_Not_Work_Pct"
        ]
    ]
    .describe()
    .T
    .round(2)
)

,count,mean,std,min,25%,50%,75%,max
Prime_Age_FTYR_Pct,3143.0,60.88,8.13,19.77,56.53,61.63,66.20,89.34
Prime_Age_Less_Than_FTYR_Pct,3143.0,20.66,4.73,0.00,17.98,20.43,23.01,55.63
Prime_Age_Did_Not_Work_Pct,3143.0,18.46,8.19,0.99,12.98,17.02,22.03,67.39


In [12]:
# Identify county with missing prime-age work-status percentages

missing_labor = labor_raw[
    labor_raw["Work_Status_Pct_Sum"].isna()
][
    [
        "FIPS",
        "NAME",
        "Prime_Age_Pop",
        "Prime_Age_FTYR",
        "Prime_Age_Less_Than_FTYR",
        "Prime_Age_Did_Not_Work"
    ]
]

display(missing_labor)

,FIPS,NAME,Prime_Age_Pop,Prime_Age_FTYR,Prime_Age_Less_Than_FTYR,Prime_Age_Did_Not_Work
2675,48301,"Loving County, Texas",0,0,0,0


### Missing-Value Note

Loving County, Texas (FIPS 48301) has a reported prime-age population
(age 25–54) of zero in ACS table B23027. Derived prime-age work-status
percentages are therefore undefined and are retained as missing rather
than assigned a value of zero.

In [18]:
# Create labor indicator analysis dataset

labor_analysis = master_final[
    [
        "FIPS",
        "State",
        "County",
        "Poverty_All_Pct",
        "Unemployment_Pct",
        "Severe_Burden_Pct",
        "Median_Household_Income",
        "Food_Insecurity_Rate"
    ]
].merge(
    labor_raw[
        [
            "FIPS",
            "Prime_Age_Pop",
            "Prime_Age_FTYR_Pct",
            "Prime_Age_Less_Than_FTYR_Pct",
            "Prime_Age_Did_Not_Work_Pct"
        ]
    ],
    on="FIPS",
    how="left"
)

print("Rows:", len(labor_analysis))
print("Unique FIPS:", labor_analysis["FIPS"].nunique())
print("Duplicate FIPS:", labor_analysis["FIPS"].duplicated().sum())

Rows: 3144
Unique FIPS: 3144
Duplicate FIPS: 0


In [19]:
# Examine relationships among labor and economic hardship measures

corr_columns = [
    "Unemployment_Pct",
    "Prime_Age_FTYR_Pct",
    "Prime_Age_Less_Than_FTYR_Pct",
    "Prime_Age_Did_Not_Work_Pct",
    "Poverty_All_Pct",
    "Severe_Burden_Pct",
    "Median_Household_Income",
    "Food_Insecurity_Rate"
]

labor_corr = (
    labor_analysis[corr_columns]
    .corr()
    .round(3)
)

display(labor_corr)

,Unemployment_Pct,Prime_Age_FTYR_Pct,Prime_Age_Less_Than_FTYR_Pct,Prime_Age_Did_Not_Work_Pct,Poverty_All_Pct,Severe_Burden_Pct,Median_Household_Income,Food_Insecurity_Rate
Unemployment_Pct,1.000,-0.502,0.201,0.382,0.400,0.143,-0.294,0.392
Prime_Age_FTYR_Pct,-0.502,1.000,-0.279,-0.832,-0.662,-0.082,0.482,-0.554
Prime_Age_Less_Than_FTYR_Pct,0.201,-0.279,1.000,-0.301,-0.089,0.141,0.102,-0.069
Prime_Age_Did_Not_Work_Pct,0.382,-0.832,-0.301,1.000,0.708,-0.000,-0.537,0.590
Poverty_All_Pct,0.400,-0.662,-0.089,0.708,1.000,0.042,-0.768,0.740
Severe_Burden_Pct,0.143,-0.082,0.141,-0.000,0.042,1.000,0.126,-0.014
Median_Household_Income,-0.294,0.482,0.102,-0.537,-0.768,0.126,1.000,-0.692
Food_Insecurity_Rate,0.392,-0.554,-0.069,0.590,0.740,-0.014,-0.692,1.000


In [20]:
# Compare selected counties across labor-market measures

counties_to_compare = [
    "22035",  # East Carroll Parish, LA
    "06025",  # Imperial County, CA
    "02158",  # Kusilvak Census Area, AK
    "46137",  # Ziebach County, SD
    "36005",  # Bronx County, NY
    "50003",  # Bennington County, VT
    "13309",  # Wheeler County, GA
    "48301"   # Loving County, TX
]

county_labor_compare = (
    labor_analysis[
        labor_analysis["FIPS"].isin(counties_to_compare)
    ][
        [
            "FIPS",
            "State",
            "County",
            "Poverty_All_Pct",
            "Unemployment_Pct",
            "Prime_Age_FTYR_Pct",
            "Prime_Age_Less_Than_FTYR_Pct",
            "Prime_Age_Did_Not_Work_Pct",
            "Median_Household_Income",
            "Food_Insecurity_Rate"
        ]
    ]
    .sort_values("Poverty_All_Pct", ascending=False)
)

display(county_labor_compare.round(2))

,FIPS,State,County,Poverty_All_Pct,Unemployment_Pct,Prime_Age_FTYR_Pct,Prime_Age_Less_Than_FTYR_Pct,Prime_Age_Did_Not_Work_Pct,Median_Household_Income,Food_Insecurity_Rate
2429,46137,SD,Ziebach County,45.8,2.9,51.54,19.31,29.14,45049.0,0.22
1132,22035,LA,East Carroll Parish,44.4,13.0,30.62,15.91,53.47,36713.0,0.27
541,13309,GA,Wheeler County,35.0,2.9,35.37,5.41,59.22,42918.0,0.19
83,02158,AK,Kusilvak Census Area,34.9,17.4,22.03,50.76,27.21,43511.0,0.26
1832,36005,NY,Bronx County,28.7,7.0,49.28,26.21,24.52,46486.0,0.27
199,06025,CA,Imperial County,18.8,18.9,47.27,23.01,29.72,60477.0,0.29
2809,50003,VT,Bennington County,9.9,3.1,58.36,24.77,16.88,71854.0,0.14
2675,48301,TX,Loving County,6.3,0.7,NaN,NaN,NaN,128055.0,NaN


## Stage 3 — Earnings Adequacy

Employment status alone does not indicate whether work provides sufficient
economic resources.

This stage evaluates county-level median earnings as an additional measure
of labor-market economic security.

Two ACS 2024 5-Year measures are collected:

- Median earnings among all people age 16+ with earnings
- Median earnings among full-time, year-round workers with earnings

The broader earnings measure is of primary interest because it includes workers
with part-time, part-year, intermittent, and other work patterns that may be
excluded from a full-time/year-round measure.

In [21]:
# ACS 2024 5-Year Subject Table S2001
# Earnings in the Past 12 Months

earnings_url = "https://api.census.gov/data/2024/acs/acs5/subject"

earnings_params = {
    "get": ",".join([
        "NAME",
        "S2001_C01_002E",  # Median earnings - all people 16+ with earnings
        "S2001_C01_013E"   # Median earnings - FT/YR workers with earnings
    ]),
    "for": "county:*",
    "in": "state:*",
    "key": CENSUS_API_KEY
}

earnings_response = requests.get(
    earnings_url,
    params=earnings_params,
    timeout=30
)

print("Status code:", earnings_response.status_code)

Status code: 200


In [22]:
# Convert earnings response to DataFrame

earnings_data = earnings_response.json()

earnings_raw = pd.DataFrame(
    earnings_data[1:],
    columns=earnings_data[0]
)

earnings_raw["FIPS"] = (
    earnings_raw["state"].str.zfill(2)
    + earnings_raw["county"].str.zfill(3)
)

earnings_raw[
    ["S2001_C01_002E", "S2001_C01_013E"]
] = earnings_raw[
    ["S2001_C01_002E", "S2001_C01_013E"]
].apply(pd.to_numeric, errors="coerce")

print("Rows:", len(earnings_raw))
print("Unique FIPS:", earnings_raw["FIPS"].nunique())
print("Duplicate FIPS:", earnings_raw["FIPS"].duplicated().sum())

Rows: 3222
Unique FIPS: 3222
Duplicate FIPS: 0


In [23]:
earnings_raw = earnings_raw[
    earnings_raw["state"] != "72"
].copy()

print("Rows after PR exclusion:", len(earnings_raw))
print("Unique FIPS:", earnings_raw["FIPS"].nunique())

Rows after PR exclusion: 3144
Unique FIPS: 3144


In [24]:
earnings_raw = earnings_raw.rename(
    columns={
        "S2001_C01_002E": "Median_Earnings_All_Workers",
        "S2001_C01_013E": "Median_Earnings_FTYR"
    }
)

In [25]:
display(
    earnings_raw[
        [
            "Median_Earnings_All_Workers",
            "Median_Earnings_FTYR"
        ]
    ]
    .describe()
    .T
    .round(2)
)

print("\nMissing values:")
display(
    earnings_raw[
        [
            "Median_Earnings_All_Workers",
            "Median_Earnings_FTYR"
        ]
    ].isna().sum()
)

,count,mean,std,min,25%,50%,75%,max
Median_Earnings_All_Workers,3144.0,-807192.23,23769325.26,-666666666.0,36446.25,40368.0,44386.50,91259.0
Median_Earnings_FTYR,3144.0,-582187.46,20588518.63,-666666666.0,48122.00,52327.5,57912.25,123559.0



Missing values:


Median_Earnings_All_Workers    0
Median_Earnings_FTYR           0
dtype: int64

In [26]:
# Inspect Census special-value records

special_earnings = earnings_raw[
    (earnings_raw["Median_Earnings_All_Workers"] < 0)
    | (earnings_raw["Median_Earnings_FTYR"] < 0)
][
    [
        "FIPS",
        "NAME",
        "Median_Earnings_All_Workers",
        "Median_Earnings_FTYR"
    ]
]

display(special_earnings)

print("Count:", len(special_earnings))

,FIPS,NAME,Median_Earnings_All_Workers,Median_Earnings_FTYR
74,02068,"Denali Borough, Alaska",68594,-666666666
95,02282,"Yakutat City and Borough, Alaska",-666666666,76250
1803,35011,"De Baca County, New Mexico",30742,-666666666
2593,48137,"Edwards County, Texas",-666666666,26029
2655,48261,"Kenedy County, Texas",-666666666,-666666666
2675,48301,"Loving County, Texas",-666666666,96250


Count: 6


In [27]:
# Replace ACS special negative values with missing

earnings_cols = [
    "Median_Earnings_All_Workers",
    "Median_Earnings_FTYR"
]

earnings_raw[earnings_cols] = (
    earnings_raw[earnings_cols]
    .mask(earnings_raw[earnings_cols] < 0)
)

In [28]:
display(
    earnings_raw[earnings_cols]
    .describe()
    .T
    .round(2)
)

print("\nMissing values:")
display(
    earnings_raw[earnings_cols]
    .isna()
    .sum()
)

,count,mean,std,min,25%,50%,75%,max
Median_Earnings_All_Workers,3140.0,41036.40,7324.56,12293.0,36450.25,40375.0,44392.75,91259.0
Median_Earnings_FTYR,3141.0,53996.38,9768.04,26029.0,48155.00,52343.0,57916.00,123559.0



Missing values:


Median_Earnings_All_Workers    4
Median_Earnings_FTYR           3
dtype: int64

### Preliminary Earnings Finding

The 2024 ACS earnings measures provide nearly complete county coverage after
Census special-value codes are treated as missing.

Median earnings among all people age 16+ with earnings are lower than median
earnings among full-time, year-round workers, illustrating that employment
status alone does not describe the economic return to work.

However, these earnings measures use a broader age universe than the prime-age
25–54 work-status measures developed from B23027. Before using earnings in a
labor-stress construct, age-aligned earnings measures should be investigated.

Published medians for separate age groups should not be averaged to construct
a combined prime-age median.

In [29]:
# Add ACS earnings measures to labor analysis dataset

labor_analysis = labor_analysis.merge(
    earnings_raw[
        [
            "FIPS",
            "Median_Earnings_All_Workers",
            "Median_Earnings_FTYR"
        ]
    ],
    on="FIPS",
    how="left"
)

print("Rows:", len(labor_analysis))
print("Unique FIPS:", labor_analysis["FIPS"].nunique())
print("Duplicate FIPS:", labor_analysis["FIPS"].duplicated().sum())

print("\nMissing earnings:")
print(
    labor_analysis[
        [
            "Median_Earnings_All_Workers",
            "Median_Earnings_FTYR"
        ]
    ].isna().sum()
)

Rows: 3144
Unique FIPS: 3144
Duplicate FIPS: 0

Missing earnings:
Median_Earnings_All_Workers    4
Median_Earnings_FTYR           3
dtype: int64


In [30]:
# Compare earnings with labor and economic hardship measures

earnings_corr_columns = [
    "Unemployment_Pct",
    "Prime_Age_FTYR_Pct",
    "Prime_Age_Less_Than_FTYR_Pct",
    "Prime_Age_Did_Not_Work_Pct",
    "Median_Earnings_All_Workers",
    "Median_Earnings_FTYR",
    "Poverty_All_Pct",
    "Severe_Burden_Pct",
    "Median_Household_Income",
    "Food_Insecurity_Rate"
]

earnings_corr = (
    labor_analysis[earnings_corr_columns]
    .corr()
    .round(3)
)

display(earnings_corr)

,Unemployment_Pct,Prime_Age_FTYR_Pct,Prime_Age_Less_Than_FTYR_Pct,Prime_Age_Did_Not_Work_Pct,Median_Earnings_All_Workers,Median_Earnings_FTYR,Poverty_All_Pct,Severe_Burden_Pct,Median_Household_Income,Food_Insecurity_Rate
Unemployment_Pct,1.000,-0.502,0.201,0.382,-0.253,-0.143,0.400,0.143,-0.294,0.392
Prime_Age_FTYR_Pct,-0.502,1.000,-0.279,-0.832,0.494,0.347,-0.662,-0.082,0.482,-0.554
Prime_Age_Less_Than_FTYR_Pct,0.201,-0.279,1.000,-0.301,-0.080,0.180,-0.089,0.141,0.102,-0.069
Prime_Age_Did_Not_Work_Pct,0.382,-0.832,-0.301,1.000,-0.445,-0.448,0.708,-0.000,-0.537,0.590
Median_Earnings_All_Workers,-0.253,0.494,-0.080,-0.445,1.000,0.876,-0.655,0.063,0.841,-0.623
Median_Earnings_FTYR,-0.143,0.347,0.180,-0.448,0.876,1.000,-0.606,0.194,0.869,-0.606
Poverty_All_Pct,0.400,-0.662,-0.089,0.708,-0.655,-0.606,1.000,0.042,-0.768,0.740
Severe_Burden_Pct,0.143,-0.082,0.141,-0.000,0.063,0.194,0.042,1.000,0.126,-0.014
Median_Household_Income,-0.294,0.482,0.102,-0.537,0.841,0.869,-0.768,0.126,1.000,-0.692
Food_Insecurity_Rate,0.392,-0.554,-0.069,0.590,-0.623,-0.606,0.740,-0.014,-0.692,1.000


## Stage 5 — Preliminary Findings: Rethinking the Labor Dimension

The initial Economic Stress Score used poverty, unemployment, and severe
housing burden as three equally weighted indicators of county-level economic
stress. Investigation of the score raised an important methodological question:
whether the official unemployment rate adequately represents labor-related
economic insecurity.

### Unemployment and Labor-Market Security

The 2025 BLS annual-average unemployment rate remains a valid and useful
labor-market indicator. However, the analysis demonstrates that low
unemployment does not necessarily indicate strong or sustained employment
among the broader working-age population.

Using ACS 2024 5-Year Detailed Table B23027, three work-status measures were
constructed for the prime working-age population (ages 25–54):

- Full-time, year-round employment
- Work that was less than full-time and/or year-round
- No work during the past 12 months

The official unemployment rate showed only a moderate relationship with the
prime-age did-not-work rate (r = 0.382) and a moderate inverse relationship
with the prime-age full-time/year-round rate (r = -0.502). These results
indicate that the measures capture related but substantially different aspects
of county labor-market conditions.

Several county examples illustrate this distinction. Wheeler County, Georgia,
had a 2025 unemployment rate of only 2.9%, while 59.22% of prime-age adults
were estimated to have performed no work during the previous 12 months.
Ziebach County, South Dakota, also had 2.9% unemployment despite a 45.8%
poverty rate and a 29.14% prime-age did-not-work rate.

These examples demonstrate why low unemployment should not automatically be
interpreted as evidence of low economic stress.

### Prime-Age Work Attachment

Prime-age work attachment showed substantially stronger relationships with
several measures of material hardship than the official unemployment rate.

The prime-age did-not-work rate was correlated with:

- Poverty: r = 0.708
- Food insecurity: r = 0.590
- Median household income: r = -0.537
- Unemployment: r = 0.382

The prime-age full-time/year-round employment rate was correlated with:

- Poverty: r = -0.662
- Food insecurity: r = -0.554
- Median household income: r = 0.482
- Unemployment: r = -0.502

These findings suggest that prime-age work attachment may provide information
about labor-related economic insecurity that is not captured by unemployment
alone.

However, the strong relationship between prime-age non-work and poverty
(r = 0.708) also raises an important index-design question. Including both
measures could represent distinct but related dimensions of hardship, or it
could unintentionally increase the influence of a common underlying dimension
of socioeconomic disadvantage. This requires further testing before either
measure is incorporated into a composite index.

### Less-Than-Full-Time/Year-Round Work

The percentage of prime-age adults who worked less than full-time and/or
year-round was retained as an important descriptive measure but does not
currently appear suitable as a direct stress indicator.

Its relationships with poverty (r = -0.089), food insecurity (r = -0.069),
and median household income (r = 0.102) were weak.

This category also represents heterogeneous circumstances. The ACS measure
does not identify whether less-than-full-time/year-round work reflects
voluntary part-time work, seasonal employment, insufficient hours,
intermittent employment, entry into or exit from employment, or other
circumstances. It therefore should not be interpreted as equivalent to
underemployment.

### Earnings

Median earnings were investigated as a possible measure of the economic return
to work.

After Census special-value codes were treated as missing, both ACS earnings
measures provided greater than 99% county coverage. Median earnings among all
workers showed meaningful relationships with poverty (r = -0.655), food
insecurity (r = -0.623), and prime-age full-time/year-round employment
(r = 0.494).

However, median earnings were also highly correlated with median household
income:

- All-worker median earnings: r = 0.841
- Full-time/year-round median earnings: r = 0.869

The two earnings measures were themselves highly correlated (r = 0.876).

These relationships suggest that earnings provide valuable descriptive
information but may substantially overlap with the broader income environment.
Raw earnings also do not account for geographic differences in purchasing
power or cost of living. Earnings will therefore remain an exploratory
indicator rather than being incorporated into the Economic Stress Index at
this stage.

### Severe Housing Burden as a Distinct Dimension

One of the clearest findings of the analysis is the relative independence of
severe housing burden from the income and labor indicators.

Severe housing burden showed correlations of approximately:

- Poverty: r = 0.042
- Prime-age did not work: r = 0.000
- Prime-age full-time/year-round work: r = -0.082
- All-worker median earnings: r = 0.063
- Food insecurity: r = -0.014

This suggests that severe housing burden captures a substantially different
dimension of county economic stress rather than reproducing poverty, income,
or labor-market conditions.

### Current Research Direction

The analysis does not support discarding the unemployment rate. Unemployment
remains an important labor-market measure and should remain available in the
County Explorer.

However, the evidence raises substantial concerns about using unemployment as
the sole labor component of a broader Economic Stress Index, particularly
because a low unemployment rate can coexist with high poverty and weak
prime-age work attachment.

The strongest alternative candidate identified so far is the prime-age
did-not-work rate. It captures labor-market detachment among the population
ages 25–54 while avoiding the assumption that low unemployment necessarily
represents economic security.

No replacement indicator has yet been selected.

The next stage will conduct a controlled comparison between:

1. Poverty + Unemployment + Severe Housing Burden
2. Poverty + Prime-Age Did-Not-Work + Severe Housing Burden

Normalization and weighting will be held constant so that the effect of
changing the labor indicator can be evaluated independently. County rankings,
component influence, extreme cases, and sensitivity to correlated indicators
will be examined before a final index design is selected.

## Stage 6 — Controlled Labor Indicator Comparison

This stage evaluates whether replacing the BLS unemployment rate with the
ACS prime-age did-not-work rate improves the representation of labor-related
economic stress within the composite index.

Two candidate structures are compared:

**Model U — Unemployment**
- Poverty
- Unemployment
- Severe housing burden

**Model N — Prime-Age Non-Work**
- Poverty
- Prime-age did-not-work rate
- Severe housing burden

All other methodological choices are held constant. Each component is
converted to a national county percentile and weighted equally.

This experiment isolates the effect of labor-indicator selection rather than
simultaneously changing normalization, weighting, or aggregation methods.

The purpose is not to select the model that produces the highest correlation
with food insecurity. The comparison will examine component balance, county
rankings, extreme cases, and whether the alternative labor measure provides
meaningful information without allowing correlated poverty and non-work
measures to dominate the index.

In [31]:
# Create percentile-normalized components for controlled comparison

labor_analysis["Poverty_Percentile"] = (
    labor_analysis["Poverty_All_Pct"]
    .rank(method="average", pct=True)
)

labor_analysis["Unemployment_Percentile"] = (
    labor_analysis["Unemployment_Pct"]
    .rank(method="average", pct=True)
)

labor_analysis["NonWork_Percentile"] = (
    labor_analysis["Prime_Age_Did_Not_Work_Pct"]
    .rank(method="average", pct=True)
)

labor_analysis["Housing_Percentile"] = (
    labor_analysis["Severe_Burden_Pct"]
    .rank(method="average", pct=True)
)

In [32]:
# Model U: Poverty + Unemployment + Severe Housing Burden

labor_analysis["ESI_Model_U"] = (
    labor_analysis[
        [
            "Poverty_Percentile",
            "Unemployment_Percentile",
            "Housing_Percentile"
        ]
    ]
    .mean(axis=1, skipna=False)
    * 100
)

# Model N: Poverty + Prime-Age Non-Work + Severe Housing Burden

labor_analysis["ESI_Model_N"] = (
    labor_analysis[
        [
            "Poverty_Percentile",
            "NonWork_Percentile",
            "Housing_Percentile"
        ]
    ]
    .mean(axis=1, skipna=False)
    * 100
)

print("Valid Model U:", labor_analysis["ESI_Model_U"].notna().sum())
print("Valid Model N:", labor_analysis["ESI_Model_N"].notna().sum())

Valid Model U: 3143
Valid Model N: 3142


In [33]:
display(
    labor_analysis[
        [
            "ESI_Model_U",
            "ESI_Model_N"
        ]
    ]
    .describe()
    .T
    .round(3)
)

,count,mean,std,min,25%,50%,75%,max
ESI_Model_U,3143.0,50.016,19.761,1.008,35.136,50.753,64.728,99.910
ESI_Model_N,3142.0,50.029,20.318,2.381,33.873,50.376,65.780,99.809


In [34]:
# Compare models only where both scores are available

model_compare = labor_analysis[
    labor_analysis[
        ["ESI_Model_U", "ESI_Model_N"]
    ].notna().all(axis=1)
].copy()

print("Counties available for direct comparison:", len(model_compare))

print(
    "Pearson score correlation:",
    round(
        model_compare["ESI_Model_U"]
        .corr(model_compare["ESI_Model_N"]),
        3
    )
)

print(
    "Spearman rank correlation:",
    round(
        model_compare["ESI_Model_U"]
        .corr(
            model_compare["ESI_Model_N"],
            method="spearman"
        ),
        3
    )
)

Counties available for direct comparison: 3142
Pearson score correlation: 0.874
Spearman rank correlation: 0.867


In [35]:
# Rank counties under each model
# Rank 1 = highest economic stress

model_compare["Model_U_Rank"] = (
    model_compare["ESI_Model_U"]
    .rank(method="min", ascending=False)
)

model_compare["Model_N_Rank"] = (
    model_compare["ESI_Model_N"]
    .rank(method="min", ascending=False)
)

# Positive Rank_Change means the county moves toward
# higher economic stress under Model N.

model_compare["Rank_Change"] = (
    model_compare["Model_U_Rank"]
    - model_compare["Model_N_Rank"]
)

model_compare["Absolute_Rank_Change"] = (
    model_compare["Rank_Change"].abs()
)

rank_changes = (
    model_compare[
        [
            "FIPS",
            "State",
            "County",
            "Poverty_All_Pct",
            "Unemployment_Pct",
            "Prime_Age_Did_Not_Work_Pct",
            "Severe_Burden_Pct",
            "ESI_Model_U",
            "ESI_Model_N",
            "Model_U_Rank",
            "Model_N_Rank",
            "Rank_Change",
            "Absolute_Rank_Change"
        ]
    ]
    .sort_values("Absolute_Rank_Change", ascending=False)
)

display(rank_changes.head(20).round(2))

,FIPS,State,County,Poverty_All_Pct,Unemployment_Pct,Prime_Age_Did_Not_Work_Pct,Severe_Burden_Pct,ESI_Model_U,ESI_Model_N,Model_U_Rank,Model_N_Rank,Rank_Change,Absolute_Rank_Change
96,02290,AK,Yukon-Koyukuk Census Area,19.9,10.0,9.16,5.74,63.64,32.89,854.0,2413.0,-1559.0,1559.0
2562,48075,TX,Childress County,20.0,2.5,41.27,16.67,44.61,75.86,1886.0,343.0,1543.0,1543.0
2366,46007,SD,Bennett County,29.4,2.6,43.39,13.39,42.77,73.81,1967.0,426.0,1541.0,1541.0
2161,40057,OK,Harmon County,24.5,2.2,31.50,16.54,46.35,76.82,1799.0,309.0,1490.0,1490.0
541,13309,GA,Wheeler County,35.0,2.9,59.22,13.49,45.15,74.85,1857.0,377.0,1480.0,1480.0
84,02164,AK,Lake and Peninsula Borough,20.2,5.8,7.49,1.57,60.18,30.72,1035.0,2511.0,-1476.0,1476.0
2134,40003,OK,Alfalfa County,18.1,2.5,28.18,13.43,36.68,65.10,2291.0,823.0,1468.0,1468.0
77,02100,AK,Haines Borough,10.1,9.0,9.89,17.45,56.76,26.87,1222.0,2681.0,-1459.0,1459.0
2410,46095,SD,Mellette County,25.9,3.0,41.57,10.93,41.83,70.11,2013.0,569.0,1444.0,1444.0
2575,48101,TX,Cottle County,19.0,2.4,26.89,8.20,31.89,59.94,2493.0,1080.0,1413.0,1413.0


## Stage 7 — Margin-of-Error Audit

Because ACS estimates are sample-based, the reliability of the prime-age
work-status measures must be evaluated before they are considered for use
in the Economic Stress Index.

The ACS publishes margins of error at the 90% confidence level. The
prime-age did-not-work rate used in this analysis is a derived measure,
constructed by combining published estimates for ages 25–44 and 45–54.

This stage first examines the published margins of error for the component
estimates, with particular attention to counties experiencing the largest
rank changes between Model U and Model N.

Because the final prime-age percentage combines multiple ACS estimates,
the component margins of error should not be interpreted as the exact
margin of error of the derived percentage.

In [36]:
# Pull B23027 estimates and margins of error
# needed for the prime-age did-not-work measure

moe_params = {
    "get": ",".join([
        "NAME",

        # Ages 25-44
        "B23027_012E",  # Total
        "B23027_012M",  # Total MOE
        "B23027_016E",  # Did not work
        "B23027_016M",  # Did not work MOE

        # Ages 45-54
        "B23027_017E",  # Total
        "B23027_017M",  # Total MOE
        "B23027_021E",  # Did not work
        "B23027_021M"   # Did not work MOE
    ]),
    "for": "county:*",
    "in": "state:*",
    "key": CENSUS_API_KEY
}

moe_response = requests.get(
    url,
    params=moe_params,
    timeout=30
)

print("Status code:", moe_response.status_code)

Status code: 200


In [37]:
# Convert MOE response to DataFrame

moe_data = moe_response.json()

labor_moe = pd.DataFrame(
    moe_data[1:],
    columns=moe_data[0]
)

labor_moe["FIPS"] = (
    labor_moe["state"].str.zfill(2)
    + labor_moe["county"].str.zfill(3)
)

# Exclude Puerto Rico to match canonical county universe

labor_moe = labor_moe[
    labor_moe["state"] != "72"
].copy()

moe_numeric_cols = [
    "B23027_012E",
    "B23027_012M",
    "B23027_016E",
    "B23027_016M",
    "B23027_017E",
    "B23027_017M",
    "B23027_021E",
    "B23027_021M"
]

labor_moe[moe_numeric_cols] = (
    labor_moe[moe_numeric_cols]
    .apply(pd.to_numeric, errors="coerce")
)

print("Rows:", len(labor_moe))
print("Unique FIPS:", labor_moe["FIPS"].nunique())
print("Duplicate FIPS:", labor_moe["FIPS"].duplicated().sum())

Rows: 3144
Unique FIPS: 3144
Duplicate FIPS: 0


In [38]:
# Attach published ACS component MOEs to largest rank-change counties

moe_audit = rank_changes.head(20).merge(
    labor_moe[
        [
            "FIPS",
            "B23027_012E",
            "B23027_012M",
            "B23027_016E",
            "B23027_016M",
            "B23027_017E",
            "B23027_017M",
            "B23027_021E",
            "B23027_021M"
        ]
    ],
    on="FIPS",
    how="left"
)

display(
    moe_audit[
        [
            "FIPS",
            "State",
            "County",
            "Prime_Age_Did_Not_Work_Pct",

            "B23027_012E",
            "B23027_012M",
            "B23027_016E",
            "B23027_016M",

            "B23027_017E",
            "B23027_017M",
            "B23027_021E",
            "B23027_021M"
        ]
    ].round(2)
)

,FIPS,State,County,Prime_Age_Did_Not_Work_Pct,B23027_012E,B23027_012M,B23027_016E,B23027_016M,B23027_017E,B23027_017M,B23027_021E,B23027_021M
0,02290,AK,Yukon-Koyukuk Census Area,9.16,1400,92,118,47,511,72,57,28
1,48075,TX,Childress County,41.27,2282,254,997,217,793,161,272,117
2,46007,SD,Bennett County,43.39,652,110,273,105,286,66,134,50
3,40057,OK,Harmon County,31.50,541,83,176,70,275,59,81,28
4,13309,GA,Wheeler County,59.22,2058,260,1286,267,622,163,301,88
5,02164,AK,Lake and Peninsula Borough,7.49,385,59,27,15,69,36,7,5
6,40003,OK,Alfalfa County,28.18,1604,85,431,83,848,63,260,71
7,02100,AK,Haines Borough,9.89,321,105,60,51,397,93,11,14
8,46095,SD,Mellette County,41.57,458,78,214,56,266,65,87,45
9,48101,TX,Cottle County,26.89,228,83,96,70,129,54,0,15


In [39]:
# Stage 7B — Check whether B23027 is available
# in the 2024 ACS 5-Year Variance Replicate Estimates

vre_table_list_url = (
    "https://www2.census.gov/programs-surveys/acs/"
    "replicate_estimates/2024/documentation/5-year/"
    "2020-2024_Variance_Replicate_Estimates_Table_List.csv"
)

vre_tables = pd.read_csv(vre_table_list_url)

print(vre_tables.columns.tolist())
print("Rows:", len(vre_tables))

display(
    vre_tables[
        vre_tables.astype(str)
        .apply(
            lambda row: row.str.contains(
                "B23027",
                case=False,
                na=False
            ).any(),
            axis=1
        )
    ]
)

HTTPError: HTTP Error 404: Not Found

### Variance Replicate Estimate Availability

The Census Bureau's 2024 ACS 5-Year Variance Replicate Estimate (VRE)
tables were investigated as a method for calculating an exact margin of
error for the derived prime-age did-not-work percentage.

VRE tables contain 80 replicate estimates and allow derived margins of
error to incorporate covariance among aggregated ACS estimates.

However, VREs are published only for selected ACS Detailed Tables.
B23027, the table used to construct the prime-age work-status measures
in this analysis, is not included among the 2024 county-level VRE tables.

Therefore, an exact replicate-based margin of error cannot be calculated
for the derived B23027 prime-age did-not-work percentage using the
published 2024 VRE products.

The analysis will instead evaluate an approximate derived margin of error
using the published ACS margins of error for the B23027 component
estimates, with this limitation documented explicitly.

In [40]:
# Stage 7B — Approximate MOE for derived prime-age non-work rate
#
# ACS published MOEs are 90% margins of error.
# Because B23027 VREs are unavailable, this uses the Census
# approximation formulas for aggregated counts and derived proportions.

import numpy as np

# Combined prime-age population, ages 25–54
labor_moe["Prime_Age_Pop_Est"] = (
    labor_moe["B23027_012E"]
    + labor_moe["B23027_017E"]
)

# Approximate MOE for combined prime-age population
labor_moe["Prime_Age_Pop_MOE"] = np.sqrt(
    labor_moe["B23027_012M"] ** 2
    + labor_moe["B23027_017M"] ** 2
)

# Combined prime-age adults who did not work
labor_moe["Prime_Age_NonWork_Est"] = (
    labor_moe["B23027_016E"]
    + labor_moe["B23027_021E"]
)

# Approximate MOE for combined non-work estimate
labor_moe["Prime_Age_NonWork_MOE"] = np.sqrt(
    labor_moe["B23027_016M"] ** 2
    + labor_moe["B23027_021M"] ** 2
)

In [41]:
# Derived prime-age non-work proportion

labor_moe["Prime_Age_NonWork_Prop"] = (
    labor_moe["Prime_Age_NonWork_Est"]
    / labor_moe["Prime_Age_Pop_Est"]
)

# Quantity inside Census derived-proportion MOE formula

labor_moe["MOE_Radicand"] = (
    labor_moe["Prime_Age_NonWork_MOE"] ** 2
    - (
        labor_moe["Prime_Age_NonWork_Prop"] ** 2
        * labor_moe["Prime_Age_Pop_MOE"] ** 2
    )
)

print(
    "Count with negative MOE radicand:",
    (labor_moe["MOE_Radicand"] < 0).sum()
)

Count with negative MOE radicand: 0


In [42]:
# Calculate approximate 90% MOE for prime-age non-work percentage

labor_moe["Prime_Age_NonWork_MOE_Approx"] = (
    np.sqrt(labor_moe["MOE_Radicand"])
    / labor_moe["Prime_Age_Pop_Est"]
    * 100
)

# Percentage point estimate

labor_moe["Prime_Age_NonWork_Pct"] = (
    labor_moe["Prime_Age_NonWork_Prop"] * 100
)

# Approximate 90% confidence interval

labor_moe["Prime_Age_NonWork_Lower_90"] = (
    labor_moe["Prime_Age_NonWork_Pct"]
    - labor_moe["Prime_Age_NonWork_MOE_Approx"]
).clip(lower=0)

labor_moe["Prime_Age_NonWork_Upper_90"] = (
    labor_moe["Prime_Age_NonWork_Pct"]
    + labor_moe["Prime_Age_NonWork_MOE_Approx"]
).clip(upper=100)

In [43]:
# Validate derived percentage against original labor measure

moe_validation = labor_analysis[
    ["FIPS", "Prime_Age_Did_Not_Work_Pct"]
].merge(
    labor_moe[
        ["FIPS", "Prime_Age_NonWork_Pct"]
    ],
    on="FIPS",
    how="left"
)

moe_validation["Difference"] = (
    moe_validation["Prime_Age_Did_Not_Work_Pct"]
    - moe_validation["Prime_Age_NonWork_Pct"]
).abs()

print(
    "Maximum percentage difference:",
    moe_validation["Difference"].max()
)

print(
    "Rows matching within 0.01 percentage points:",
    (moe_validation["Difference"] < 0.01).sum(),
    "of",
    moe_validation["Difference"].notna().sum()
)

Maximum percentage difference: 0.0
Rows matching within 0.01 percentage points: 3143 of 3143


In [44]:
# Audit uncertainty for counties with largest Model U vs Model N rank changes

rank_moe_audit = rank_changes.head(20).merge(
    labor_moe[
        [
            "FIPS",
            "Prime_Age_Pop_Est",
            "Prime_Age_NonWork_Pct",
            "Prime_Age_NonWork_MOE_Approx",
            "Prime_Age_NonWork_Lower_90",
            "Prime_Age_NonWork_Upper_90"
        ]
    ],
    on="FIPS",
    how="left"
)

display(
    rank_moe_audit[
        [
            "FIPS",
            "State",
            "County",
            "Prime_Age_Pop_Est",
            "Unemployment_Pct",
            "Prime_Age_NonWork_Pct",
            "Prime_Age_NonWork_MOE_Approx",
            "Prime_Age_NonWork_Lower_90",
            "Prime_Age_NonWork_Upper_90",
            "Rank_Change"
        ]
    ].round(2)
)

,FIPS,State,County,Prime_Age_Pop_Est,Unemployment_Pct,Prime_Age_NonWork_Pct,Prime_Age_NonWork_MOE_Approx,Prime_Age_NonWork_Lower_90,Prime_Age_NonWork_Upper_90,Rank_Change
0,02290,AK,Yukon-Koyukuk Census Area,1911,10.0,9.16,2.81,6.35,11.97,-1559.0
1,48075,TX,Childress County,3075,2.5,41.27,6.93,34.34,48.20,1543.0
2,46007,SD,Bennett County,938,2.6,43.39,10.89,32.50,54.28,1541.0
3,40057,OK,Harmon County,816,2.2,31.50,8.36,23.13,39.86,1490.0
4,13309,GA,Wheeler County,2680,2.9,59.22,8.00,51.21,67.22,1480.0
5,02164,AK,Lake and Peninsula Borough,454,5.8,7.49,3.29,4.20,10.78,-1476.0
6,40003,OK,Alfalfa County,2452,2.5,28.18,4.29,23.90,32.47,1468.0
7,02100,AK,Haines Borough,718,9.0,9.89,7.11,2.78,17.00,-1459.0
8,46095,SD,Mellette County,724,3.0,41.57,8.03,33.55,49.60,1444.0
9,48101,TX,Cottle County,357,2.4,26.89,18.61,8.28,45.51,1413.0


## Stage 8 — Component Balance and Sensitivity Analysis

Model N replaces the unemployment component of the original Economic Stress
Index structure with the prime-age did-not-work rate.

The candidate index therefore contains three equally weighted components:

- Poverty
- Prime-age non-work
- Severe housing burden

Previous analysis found a strong positive relationship between poverty and
prime-age non-work (r = 0.708), while severe housing burden showed little
relationship with either measure.

This stage evaluates whether equal weighting nevertheless allows poverty and
prime-age non-work to exert disproportionate combined influence on the
composite.

The analysis first examines component relationships with the resulting index
and identifies the primary component driving each county's score. Sensitivity
tests will then evaluate whether reasonable alternative weighting structures
materially change county scores and rankings.

The purpose is not to optimize the index against food insecurity or another
outcome. The goal is to determine whether the candidate index is stable,
interpretable, and consistent with the intended multidimensional construct.

In [45]:
# Stage 8A — Examine relationships between Model N and its components

model_n_components = [
    "Poverty_Percentile",
    "NonWork_Percentile",
    "Housing_Percentile",
    "ESI_Model_N"
]

model_n_corr = (
    labor_analysis[model_n_components]
    .corr()
    .round(3)
)

display(model_n_corr)

,Poverty_Percentile,NonWork_Percentile,Housing_Percentile,ESI_Model_N
Poverty_Percentile,1.000,0.713,0.017,0.819
NonWork_Percentile,0.713,1.000,-0.000,0.811
Housing_Percentile,0.017,-0.000,1.000,0.481
ESI_Model_N,0.819,0.811,0.481,1.000


In [46]:
# Identify highest-stress component for each county in Model N

component_percentiles = labor_analysis[
    [
        "Poverty_Percentile",
        "NonWork_Percentile",
        "Housing_Percentile"
    ]
].copy()

component_percentiles.columns = [
    "Poverty",
    "Prime-Age Non-Work",
    "Severe Housing Burden"
]

valid_components = component_percentiles.dropna()

primary_driver_n = valid_components.idxmax(axis=1)

print("Primary Model N stress driver:")
display(primary_driver_n.value_counts())

print("\nPercent of counties:")
display(
    (
        primary_driver_n.value_counts(normalize=True) * 100
    ).round(1)
)

Primary Model N stress driver:


Severe Housing Burden    1334
Prime-Age Non-Work        905
Poverty                   903
Name: count, dtype: int64


Percent of counties:


Severe Housing Burden    42.5
Prime-Age Non-Work       28.8
Poverty                  28.7
Name: proportion, dtype: float64

In [47]:
# Examine overlap among high-stress components
# High stress is defined here only for diagnostic purposes
# as the national 75th percentile or above.

high_poverty = labor_analysis["Poverty_Percentile"] >= 0.75
high_nonwork = labor_analysis["NonWork_Percentile"] >= 0.75
high_housing = labor_analysis["Housing_Percentile"] >= 0.75

print(
    "High poverty AND high non-work:",
    (high_poverty & high_nonwork).sum()
)

print(
    "High poverty AND high housing:",
    (high_poverty & high_housing).sum()
)

print(
    "High non-work AND high housing:",
    (high_nonwork & high_housing).sum()
)

print(
    "High on all three:",
    (high_poverty & high_nonwork & high_housing).sum()
)

High poverty AND high non-work: 526
High poverty AND high housing: 225
High non-work AND high housing: 157
High on all three: 114


### Stage 8B — Weight Sensitivity

Equal weighting provides a transparent baseline but does not guarantee equal
effective influence when index components are correlated.

Poverty and prime-age non-work are strongly related, while severe housing
burden is largely independent of both. Therefore, two modest alternative
weighting structures are compared with the equal-weight model.

The alternatives are not proposed as optimized models. They represent
reasonable perturbations designed to test whether county scores and rankings
are highly sensitive to the weighting decision.

- Equal: Poverty 33.3%, Non-Work 33.3%, Housing 33.3%
- Housing40: Poverty 30%, Non-Work 30%, Housing 40%
- Poverty40: Poverty 40%, Non-Work 30%, Housing 30%

In [48]:
# Stage 8B — Weight sensitivity models

sensitivity = labor_analysis[
    [
        "FIPS",
        "State",
        "County",
        "Poverty_Percentile",
        "NonWork_Percentile",
        "Housing_Percentile"
    ]
].dropna().copy()

# Equal weighting

sensitivity["ESI_Equal"] = (
    (
        sensitivity["Poverty_Percentile"]
        + sensitivity["NonWork_Percentile"]
        + sensitivity["Housing_Percentile"]
    )
    / 3
    * 100
)

# Modestly increase independent housing dimension

sensitivity["ESI_Housing40"] = (
    (
        0.30 * sensitivity["Poverty_Percentile"]
        + 0.30 * sensitivity["NonWork_Percentile"]
        + 0.40 * sensitivity["Housing_Percentile"]
    )
    * 100
)

# Modestly increase poverty dimension

sensitivity["ESI_Poverty40"] = (
    (
        0.40 * sensitivity["Poverty_Percentile"]
        + 0.30 * sensitivity["NonWork_Percentile"]
        + 0.30 * sensitivity["Housing_Percentile"]
    )
    * 100
)

print("Counties:", len(sensitivity))

display(
    sensitivity[
        [
            "ESI_Equal",
            "ESI_Housing40",
            "ESI_Poverty40"
        ]
    ]
    .describe()
    .T
    .round(3)
)

Counties: 3142


,count,mean,std,min,25%,50%,75%,max
ESI_Equal,3142.0,50.029,20.318,2.381,33.873,50.376,65.780,99.809
ESI_Housing40,3142.0,50.030,19.837,2.286,35.166,50.329,64.505,99.822
ESI_Poverty40,3142.0,50.029,20.716,2.157,32.972,50.212,66.464,99.822


In [49]:
for score in [
    "ESI_Equal",
    "ESI_Housing40",
    "ESI_Poverty40"
]:
    sensitivity[f"{score}_Rank"] = (
        sensitivity[score]
        .rank(method="min", ascending=False)
    )

print("Spearman rank correlations:")

display(
    sensitivity[
        [
            "ESI_Equal",
            "ESI_Housing40",
            "ESI_Poverty40"
        ]
    ]
    .corr(method="spearman")
    .round(4)
)

Spearman rank correlations:


,ESI_Equal,ESI_Housing40,ESI_Poverty40
ESI_Equal,1.0000,0.991,0.9968
ESI_Housing40,0.9910,1.000,0.9800
ESI_Poverty40,0.9968,0.980,1.0000


In [50]:
# Measure rank sensitivity to alternative weighting structures

sensitivity["Housing40_Rank_Change"] = (
    sensitivity["ESI_Equal_Rank"]
    - sensitivity["ESI_Housing40_Rank"]
)

sensitivity["Poverty40_Rank_Change"] = (
    sensitivity["ESI_Equal_Rank"]
    - sensitivity["ESI_Poverty40_Rank"]
)

sensitivity["Max_Absolute_Rank_Change"] = (
    sensitivity[
        [
            "Housing40_Rank_Change",
            "Poverty40_Rank_Change"
        ]
    ]
    .abs()
    .max(axis=1)
)

print("Rank-change summary:")
display(
    sensitivity[
        [
            "Housing40_Rank_Change",
            "Poverty40_Rank_Change",
            "Max_Absolute_Rank_Change"
        ]
    ]
    .describe()
    .T
    .round(1)
)

Rank-change summary:


,count,mean,std,min,25%,50%,75%,max
Housing40_Rank_Change,3142.0,-0.1,122.0,-300.0,-78.0,1.0,80.0,293.0
Poverty40_Rank_Change,3142.0,-0.1,72.3,-318.0,-45.8,0.0,43.0,304.0
Max_Absolute_Rank_Change,3142.0,104.5,72.2,0.0,44.0,90.0,157.0,318.0


In [51]:
# Counties most sensitive to weighting choice

weight_sensitive_counties = (
    sensitivity[
        [
            "FIPS",
            "State",
            "County",
            "Poverty_Percentile",
            "NonWork_Percentile",
            "Housing_Percentile",
            "ESI_Equal",
            "ESI_Housing40",
            "ESI_Poverty40",
            "ESI_Equal_Rank",
            "ESI_Housing40_Rank",
            "ESI_Poverty40_Rank",
            "Housing40_Rank_Change",
            "Poverty40_Rank_Change",
            "Max_Absolute_Rank_Change"
        ]
    ]
    .sort_values(
        "Max_Absolute_Rank_Change",
        ascending=False
    )
)

display(weight_sensitive_counties.head(20).round(2))

,FIPS,State,County,Poverty_Percentile,NonWork_Percentile,Housing_Percentile,ESI_Equal,ESI_Housing40,ESI_Poverty40,ESI_Equal_Rank,ESI_Housing40_Rank,ESI_Poverty40_Rank,Housing40_Rank_Change,Poverty40_Rank_Change,Max_Absolute_Rank_Change
381,12119,FL,Sumter County,0.14,0.97,0.95,68.59,71.27,63.09,644.0,489.0,962.0,155.0,-318.0,318.0
84,02164,AK,Lake and Peninsula Borough,0.88,0.03,0.01,30.72,27.78,36.44,2511.0,2667.0,2207.0,-156.0,304.0,304.0
506,13239,GA,Quitman County,0.93,0.86,0.00,59.67,53.75,62.98,1094.0,1394.0,965.0,-300.0,129.0,300.0
2590,48131,TX,Duval County,0.96,0.95,0.02,64.28,58.02,67.42,866.0,1165.0,736.0,-299.0,130.0,299.0
491,13209,GA,Montgomery County,0.89,0.88,0.04,60.61,54.98,63.48,1046.0,1344.0,946.0,-298.0,100.0,298.0
2272,42053,PA,Forest County,0.94,1.00,0.03,65.34,59.07,68.16,806.0,1103.0,701.0,-297.0,105.0,297.0
2655,48261,TX,Kenedy County,0.68,0.91,0.00,52.99,47.73,54.47,1431.0,1726.0,1358.0,-295.0,73.0,295.0
2579,48109,TX,Culberson County,0.81,0.94,0.02,59.17,53.46,61.36,1118.0,1413.0,1035.0,-295.0,83.0,295.0
2809,50003,VT,Bennington County,0.22,0.49,0.99,56.83,61.08,53.34,1253.0,960.0,1417.0,293.0,-164.0,293.0
3002,54017,WV,Doddridge County,0.85,0.98,0.03,61.94,56.06,64.21,977.0,1269.0,909.0,-292.0,68.0,292.0


## Stage 8 Conclusion — Final ESI-C Specification

The component balance and sensitivity analyses support adoption of the
candidate Model N specification as the revised **Economic Stress Index —
County (ESI-C)**.

### Final Components

ESI-C combines three county-level indicators:

1. **Poverty** — county poverty rate
2. **Prime-Age Non-Work** — percentage of adults ages 25–54 who did not
   work during the reference period
3. **Severe Housing Burden** — percentage of renter households paying
   50% or more of household income toward rent

Each indicator is converted to its national county percentile rank and
receives equal weight in the final index.

The final specification is:

**ESI-C = 100 × [(Poverty Percentile + Prime-Age Non-Work Percentile +
Severe Housing Burden Percentile) / 3]**

ESI-C is therefore a **relative index**. It describes a county's position
within the national distribution of county economic stress conditions.
A score should not be interpreted as a percentage of economic stress or
as a ratio-scale measure.

### Why Unemployment Was Replaced

The original Economic Stress Score used poverty, unemployment, and severe
housing burden. Further investigation showed that the official unemployment
rate captures only one aspect of labor-market conditions and can understate
economic hardship in places where unemployment is low but substantial
portions of the prime-age population are not working.

County-level comparison demonstrated meaningful differences between
unemployment and prime-age non-work. Several counties with very low
unemployment rates had high levels of prime-age non-work, while other
counties with relatively high unemployment retained strong annual work
attachment.

Replacing unemployment with prime-age non-work therefore changes the labor
component from a measure of current unemployment among labor-force
participants to a broader measure of prime-age work detachment.

### ACS Uncertainty Audit

Because prime-age non-work is derived from ACS estimates, sampling
uncertainty was explicitly investigated before adopting the indicator.

Published ACS margins of error were collected for the B23027 component
estimates used to construct the prime-age measure. The Census Bureau's
Variance Replicate Estimate products were also investigated, but B23027
is not available among the selected 2024 county-level VRE tables.

An approximate 90% margin of error was therefore calculated for the derived
prime-age non-work percentage using published ACS component margins of error
and Census approximation methods.

The audit confirmed that uncertainty can be substantial in some small
counties and that exact percentile positions for these counties should be
interpreted cautiously. However, sampling uncertainty did not explain the
broader differences between unemployment and prime-age non-work.

Several large or comparatively well-measured counties also showed substantial
differences between the two labor indicators. In addition, extreme
prime-age non-work estimates in counties such as Wheeler County, Georgia,
and Calhoun County, Georgia remained high across their approximate 90%
confidence intervals.

ACS uncertainty is therefore treated as information about **measurement
precision**, rather than as a reason to alter the substantive index value.
Where appropriate, uncertainty should be presented alongside ESI-C in
future analytical products.

### Component Balance

The percentile-normalized ESI-C components were not equally independent.

Poverty and prime-age non-work were strongly related:

- Poverty ↔ Prime-Age Non-Work: **r = 0.713**

Severe housing burden was essentially independent of both:

- Poverty ↔ Severe Housing Burden: **r = 0.017**
- Prime-Age Non-Work ↔ Severe Housing Burden: **r ≈ 0.000**

As a result, poverty and prime-age non-work frequently reinforce one another
within the composite.

However, severe housing burden remained an important independent dimension.
It was the highest-stress component in **42.5% of counties**, compared with
28.8% for prime-age non-work and 28.7% for poverty.

Top-quartile overlap further demonstrated the clustering of poverty and
prime-age non-work:

- High poverty + high non-work: **526 counties**
- High poverty + high housing burden: **225 counties**
- High non-work + high housing burden: **157 counties**
- High on all three components: **114 counties**

These results indicate that ESI-C contains a related income/labor-hardship
cluster together with a substantially independent housing-cost pressure
dimension.

### Weight Sensitivity

Equal weighting was tested against two reasonable alternative structures:

- **Equal:** Poverty 33.3%, Non-Work 33.3%, Housing 33.3%
- **Housing40:** Poverty 30%, Non-Work 30%, Housing 40%
- **Poverty40:** Poverty 40%, Non-Work 30%, Housing 30%

County rankings remained highly stable:

- Equal ↔ Housing40: **Spearman ρ = 0.9910**
- Equal ↔ Poverty40: **Spearman ρ = 0.9968**
- Housing40 ↔ Poverty40: **Spearman ρ = 0.9800**

The median maximum county rank change across the alternative weighting
structures was approximately **90 positions** among 3,142 comparable
counties. The largest observed change was 318 positions.

Counties experiencing the greatest movement generally had strongly
asymmetric component profiles. For example, counties with very high housing
burden but relatively low poverty moved upward when housing received
additional weight, while counties with high poverty and non-work but very
low housing burden moved downward.

This behavior is substantively expected rather than evidence of general
index instability.

### Final Weighting Decision

Equal weighting is retained for ESI-C.

The decision is not based on an assumption that the three components are
equally independent or equally predictive of another outcome. Instead,
equal weighting provides the most transparent and reproducible specification,
while sensitivity testing demonstrates that reasonable alternative weights
do not materially alter the overall national pattern.

Weights were deliberately **not optimized against food insecurity** or
another external outcome. ESI-C is intended to represent county economic
stress as an independent construct, allowing its relationship with food
insecurity and other outcomes to be evaluated subsequently.

### Interpretation

ESI-C represents three distinct, though not necessarily independent,
dimensions of county economic conditions:

- material income hardship,
- prime-age work detachment, and
- severe housing cost pressure.

The index is intended for comparative county-level analysis rather than as
an absolute measure of household hardship.

The original **Economic Stress Score** is retained as the historical
class-project measure. The revised percentile-based specification developed
and tested in this notebook is designated **Economic Stress Index — County
(ESI-C)**.

### Scope and Future Research

ESI-C is intentionally parsimonious and does not attempt to capture every
dimension of local economic conditions.

Potential areas for future research include labor-force participation,
underemployment, earnings quality, local cost of living, retirement-age
population structure, seasonal employment, second-home communities, and
other factors that may affect the interpretation of county economic
conditions.

These factors were not added to the present index because the objective was
to construct a transparent and interpretable measure from a limited set of
conceptually meaningful indicators rather than maximize explanatory
coverage.

Future research may examine these factors separately or evaluate whether
they support complementary indices, including a dedicated measure of labor
market stress.